# 02 · Reliability and logging

Real API calls fail: rate limits (429), timeouts and dropped connections.
In this notebook you will:

1. Write a function that **retries** the errors worth retrying
2. See it work by calling an address that doesn't exist
3. **Log** every call to a JSON Lines file
4. Read the log back and summarise it

In [ ]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

## Step 1 · A retry function

- **Retry:** 429 rate limit, timeouts, connection errors
- **Don't retry:** 400, 401, 404. Fix the request instead.
- **Back off:** wait 2, 4, 8 seconds

The SDK already retries twice by default. We set `max_retries=0` so we can see our own retries happen.

In [ ]:
from openai import RateLimitError, APITimeoutError, APIConnectionError

RETRYABLE = (RateLimitError, APITimeoutError, APIConnectionError)


def ask(client, model, messages, retries=3, timeout=30):
    """Call the model, retrying only the errors that are worth retrying."""
    for attempt in range(1, retries + 1):
        try:
            return client.chat.completions.create(
                model=model, messages=messages, timeout=timeout
            )
        except RETRYABLE as e:
            wait = 2 ** attempt
            print(f"Retry {attempt} in {wait}s: {type(e).__name__}")
            time.sleep(wait)
    raise RuntimeError("All retries failed")


client = OpenAI(api_key=os.getenv("GEMINI_API_KEY"), base_url=GEMINI_BASE_URL, max_retries=0)

r = ask(client, GEMINI_MODEL, [{"role": "user", "content": "Say hello in Urdu and English."}])
print(r.choices[0].message.content)

## Step 2 · Watch it fail safely

This client points at an address where nothing is running, so every attempt fails with a connection error. It takes about 14 seconds.

In [ ]:
broken = OpenAI(api_key="not-needed", base_url="http://127.0.0.1:9/v1", max_retries=0)

try:
    ask(broken, GEMINI_MODEL, [{"role": "user", "content": "hi"}])
except RuntimeError as e:
    print("Gave up:", e)

## Step 3 · Log every call

One JSON object per line. **Never log API keys or personal data.**

In [ ]:
import json
import datetime
from pathlib import Path

LOG_FILE = Path("logs/calls.jsonl")
LOG_FILE.parent.mkdir(exist_ok=True)


def log_call(model, usage, latency, ok=True, error=None):
    record = {
        "time": datetime.datetime.now().isoformat(timespec="seconds"),
        "model": model,
        "tokens_in": usage.prompt_tokens if usage else None,
        "tokens_out": usage.completion_tokens if usage else None,
        "latency_s": round(latency, 2),
        "ok": ok,
        "error": error,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


def ask_and_log(client, model, messages):
    start = time.perf_counter()
    try:
        r = ask(client, model, messages)
        log_call(model, r.usage, time.perf_counter() - start)
        return r.choices[0].message.content
    except Exception as e:
        log_call(model, None, time.perf_counter() - start, ok=False, error=type(e).__name__)
        raise

In [ ]:
questions = [
    "What is a virtual environment? One sentence.",
    "What does HTTP status 429 mean? One sentence.",
    "What is JSON? One sentence.",
]

for q in questions:
    print("Q:", q)
    print("A:", ask_and_log(client, GEMINI_MODEL, [{"role": "user", "content": q}]), "\n")

## Step 4 · Read the log back

In [ ]:
records = [json.loads(line) for line in LOG_FILE.read_text().splitlines()]

ok_calls = [r for r in records if r["ok"]]
print("Calls logged:  ", len(records))
print("Successful:    ", len(ok_calls))
if ok_calls:
    print("Avg latency:   ", round(sum(r["latency_s"] for r in ok_calls) / len(ok_calls), 2), "s")
    print("Total tokens in: ", sum(r["tokens_in"] or 0 for r in ok_calls))
    print("Total tokens out:", sum(r["tokens_out"] or 0 for r in ok_calls))

print("\nLast record:", records[-1])

## Try it yourself

1. Run Step 3 with `groq_client` and `GROQ_MODEL` too. Which provider is faster in your log?
2. Add a `"provider"` field to each log record.
3. Why do we *not* retry a 401 error?